# 03 — GT-ADF Model Training

This notebook trains GT-ADF on CICIDS2017 and reproduces the paper's key results.

In [ ]:
import sys
sys.path.insert(0, '..')
import torch
from torch_geometric.loader import DataLoader
from src.data.dataset import load_dataset
from src.models.gt_adf import GTADF
from src.training.trainer import Trainer
from src.evaluation.metrics import evaluate_model, print_metrics_table
from src.utils.helpers import set_seed, get_device, model_summary

set_seed(42)
device = get_device()
print(f'Device: {device}')

In [ ]:
# Load datasets — update data_dir to your path
DATA_DIR = '../data/raw/CICIDS2017'
train_ds = load_dataset('cicids2017', DATA_DIR, split='train', binary=False)
test_ds  = load_dataset('cicids2017', DATA_DIR, split='test',  binary=False)
print(f'Train: {len(train_ds)} graphs | Test: {len(test_ds)} graphs')
print(f'Feature dim: {train_ds[0].x.shape[-1]}')

In [ ]:
val_size = max(1, int(len(train_ds) * 0.2))
train_sub, val_sub = torch.utils.data.random_split(
    train_ds, [len(train_ds) - val_size, val_size]
)
train_loader = DataLoader(train_sub, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_sub,   batch_size=64)
test_loader  = DataLoader(test_ds,   batch_size=64)

In [ ]:
in_channels = train_ds[0].x.shape[-1]
model = GTADF(
    in_channels=in_channels,
    hidden_channels=128,
    out_channels=15,    # 15 classes in CICIDS2017
    num_layers=4,
    num_heads=8,
    dropout=0.2,
    lambda_unsup=0.5,
)
model_summary(model)

In [ ]:
config = dict(
    lr=0.001, weight_decay=1e-5, batch_size=64,
    num_classes=15, lambda_unsup=0.5
)
trainer = Trainer(model, config, device=device, output_dir='../results/cicids2017')
history = trainer.train(
    train_loader, val_loader,
    max_epochs=100, patience=10, labeled_ratio=0.2
)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_title('Loss')
axes[0].legend()
axes[1].plot(history['val_accuracy'], label='Accuracy')
axes[1].plot(history['val_f1'], label='F1-Score')
axes[1].set_title('Validation Metrics')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Load best checkpoint and evaluate
from src.utils.helpers import load_checkpoint
load_checkpoint(model, '../results/cicids2017/checkpoints/gt_adf_best.pt', device=device)
metrics = evaluate_model(model, test_loader, device=device)
print_metrics_table(metrics, 'CICIDS2017')